## OKAERTool configuration
### Board: XEM6310 Spartan-6

In [1]:
import sys
import os

# Add the parent directory to the path to import pyOKAERTool (only if the package is not installed)
# sys.path.insert(0, os.path.abspath('..'))
import pyOKAERTool as okt
from pyNAVIS import *

# Define bitfile path
bitfile_path = '../bitfiles/CNAS_okaertool_XEM6310.bit'
# bitfile_path = None  # Set to None if no .bit file is to be used

# Validate the existence of the .bit file
if bitfile_path is None:
    None
elif not os.path.exists(bitfile_path):
    print(f"El archivo .bit no existe en la ruta especificada: {bitfile_path}")
    sys.exit(1)

# Create a new intance of the OkaerTool class and initialize it
okaer = okt.Okaertool(bit_file=bitfile_path)
okaer.init()

# Create a new instance of the PyNAVIS class
settings = MainSettings(num_channels=64, mono_stereo=1, on_off_both=1, address_size=4, ts_tick=0.01, bin_size=10000)

07/04/26 09:23:38 PM - INFO : Device product ID: 22, product name: XEM6310-LX150, USB speed: 2,
07/04/26 09:23:38 PM - INFO : USB 2.0 HighSpeed. USB block size set to 1 KB
07/04/26 09:23:38 PM - INFO : okaertool initialized as idle


## NAS configuration

In [2]:
import re
import os

config_file_path = '../CFBank_64_20_22000.vhd'

def _tok_to_int(tok):
    tok = tok.strip().rstrip(',').strip()
    if tok.lower().startswith('x"') and tok.endswith('"'):
        return int(tok[2:-1], 16)
    if tok.lower().startswith('0x'):
        return int(tok, 16)
    m = re.match(r'16#([0-9A-Fa-f]+)#', tok)
    if m:
        return int(m.group(1), 16)
    if tok.isdigit():
        return int(tok, 10)
    raise ValueError(f"Unrecognized token: {tok!r}")

def parse_cascade_vhd(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    text = open(path, 'r', encoding='utf-8', errors='ignore').read()

    # Find successive groups of the four parameters in the file order
    pattern = re.compile(
        r'FREQ_DIV\s*=>\s*(?P<f>[^,\n;]+)\s*,\s*'
        r'SPIKES_DIV_FB\s*=>\s*(?P<fb>[^,\n;]+)\s*,\s*'
        r'SPIKES_DIV_OUT\s*=>\s*(?P<out>[^,\n;]+)\s*,\s*'
        r'SPIKES_DIV_BPF\s*=>\s*(?P<bpf>[^,\n;]+)',
        re.IGNORECASE | re.DOTALL
    )

    values = []
    for m in pattern.finditer(text):
        f = _tok_to_int(m.group('f'))
        fb = _tok_to_int(m.group('fb'))
        out = _tok_to_int(m.group('out'))
        bpf = _tok_to_int(m.group('bpf'))
        values.extend([f, fb, out, bpf])

    return values

def reset_and_configure_okaer():
    #Reset the OkaerTool
    okaer.reset_board(mode='internal')

    # Configure the PDM2Spikes (left and right) for both NAS
    register_address = 0x0000
    okaer.logger.info("Configuring PDM2Spikes modules")
    # Left cochlea
    okaer.logger.info("Left cochlea")
    for value in PDM2Spikes_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        # okaer.set_config('port_b', register_address, value)
        register_address += 1
    # Right cochlea
    okaer.logger.info("Right cochlea")
    for value in PDM2Spikes_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        # okaer.set_config('port_b', register_address, value)
        register_address += 1

    register_address = 0x08
    okaer.logger.info("Configuring I2S2Spikes modules")
    # Configure I2S2Spikes modules for both NAS
    for value in I2S2Spikes_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        # okaer.set_config('port_b', register_address, value)

    # Configure the filters for CASCADE NAS
    okaer.logger.info("Configuring filters for Cascade NAS")
    # Left cochlea
    register_address = 0x09
    okaer.logger.info("Left cochlea")
    for value in CASCADE_FILTER_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        register_address += 1
        # # Config only 32 filters
        # if register_address >= 0x09 + 32*4:
        #     break
    # Right cochlea
    register_address = 0x010D
    okaer.logger.info("Right cochlea")
    for value in CASCADE_FILTER_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        register_address += 1
        # # Config only 32 filters
        # if register_address >= 0x010D + 32*4:
        #     break

# Define default parameters for the filters
PDM2Spikes_DEFAULT_parameter = [0x0005, 0x0006, 0x734B, 0x39C8]
I2S2Spikes_DEFAULT_parameter = [0x000F]
CASCADE_FILTER_DEFAULT_parameter = parse_cascade_vhd(config_file_path)

# quick validation / pretty print
filters = len(CASCADE_FILTER_DEFAULT_parameter) // 4
print(f"Parsed {filters} filters ({len(CASCADE_FILTER_DEFAULT_parameter)} values).")
print("CASCADE_FILTER_DEFAULT_parameter = [")
for v in CASCADE_FILTER_DEFAULT_parameter:
    # print as hex literal (4 hex digits minimum)
    width = max(2, (v.bit_length() + 3) // 4)
    print(f"    0x{v:0{width}X},")
print("]")

reset_and_configure_okaer()

07/04/26 09:23:47 PM - INFO : Board reset in mode: internal
07/04/26 09:23:47 PM - INFO : Configuring PDM2Spikes modules
07/04/26 09:23:47 PM - INFO : Left cochlea
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x0 and value 0x5
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x1 and value 0x6
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x2 and value 0x734b
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x3 and value 0x39c8
07/04/26 09:23:47 PM - INFO : Right cochlea
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x4 and value 0x5
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x5 and value 0x6
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x6 and value 0x734b
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x7 and value 0x39c8
07/04/26 09:23:47 PM - INFO : Configuring I2S2Spikes modules
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x8 and value 0xf

Parsed 65 filters (260 values).
CASCADE_FILTER_DEFAULT_parameter = [
    0x04,
    0x7CB1,
    0x7CB1,
    0x2025,
    0x04,
    0x6F93,
    0x6F93,
    0x2025,
    0x02,
    0x77CE,
    0x77CE,
    0x2025,
    0x02,
    0x6B33,
    0x6B33,
    0x2025,
    0x03,
    0x7FE5,
    0x7FE5,
    0x2025,
    0x03,
    0x7271,
    0x7271,
    0x2025,
    0x03,
    0x6666,
    0x6666,
    0x2025,
    0x04,
    0x7289,
    0x7289,
    0x2025,
    0x02,
    0x7AFB,
    0x7AFB,
    0x2025,
    0x02,
    0x6E0B,
    0x6E0B,
    0x2025,
    0x02,
    0x6277,
    0x6277,
    0x2025,
    0x03,
    0x757A,
    0x757A,
    0x2025,
    0x03,
    0x691E,
    0x691E,
    0x2025,
    0x04,
    0x7593,
    0x7593,
    0x2025,
    0x02,
    0x7E3F,
    0x7E3F,
    0x2025,
    0x02,
    0x70F7,
    0x70F7,
    0x2025,
    0x02,
    0x6514,
    0x6514,
    0x2025,
    0x03,
    0x7898,
    0x7898,
    0x2025,
    0x03,
    0x6BE8,
    0x6BE8,
    0x2025,
    0x04,
    0x78B1,
    0x78B1,
    0x2025,
    0x04,
 

07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x3c and value 0x2025
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x3d and value 0x4
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x3e and value 0x7593
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x3f and value 0x7593
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x40 and value 0x2025
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x41 and value 0x2
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x42 and value 0x7e3f
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x43 and value 0x7e3f
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x44 and value 0x2025
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x45 and value 0x2
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x46 and value 0x70f7
07/04/26 09:23:47 PM - INFO : Configuring port_a with address 0x47 and value 0x70f7
0

## External functions

In [ ]:
import threading

from colorama import Fore
import colorama
import sounddevice as sd
import soundfile as sf
import tkinter as tk
from pathlib import Path
from tkinter import filedialog

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import correlate

import AERzip

colorama.init(autoreset=True)

# --- Functions to read wav files ---
def select_wav_folder():
    root = tk.Tk()
    root.withdraw()  # Hide the main window
    root.attributes('-topmost', True)
    folder_path = filedialog.askdirectory(title="Select a folder containing WAV files", parent=root)
    root.destroy()
    return folder_path


def collect_wav_files(folder_path):
    folder = Path(folder_path)
    return sorted(str(path) for path in folder.rglob('*.wav'))


def get_wav_header_info(file_path):
    """Extracts metadata (features) from the WAV header."""
    with sf.SoundFile(file_path) as f:
        info = {
            "samplerate": f.samplerate,
            "channels": f.channels,
            "subtype": f.subtype,      # Bit depth (e.g., PCM_16)
            "format": f.format,        # File format (WAV, FLAC, etc.)
            "frames": f.frames,        # Total number of audio samples
            "duration_sec": f.frames / f.samplerate
        }
    return info

# --- Functions to trim audio for shorter playback and spike monitoring ---
plot_threads = None
plot_lock = None

def cross_correlate(recorded_signal, chirp_signal, mode="full"):
    # FFT-based correlation is usually much faster for long signals.
    correlation = correlate(recorded_signal, chirp_signal, mode=mode, method="fft")

    # Find the index of the maximum correlation value.
    max_corr_index = np.argmax(correlation)

    # Calculate the start index of the chirp in the recorded signal.
    chirp_start_index = max_corr_index - len(chirp_signal) + 1

    return chirp_start_index


def find_chirp_end(recording_file_path, chirp_file_path):
    # Load the recorded signal
    recorded_signal, recorded_sample_rate = sf.read(recording_file_path)
    chirp_signal, chirp_sample_rate = sf.read(chirp_file_path)

    recorded_signal = np.asarray(recorded_signal)
    if recorded_signal.ndim == 1:
        recorded_signal = recorded_signal[:, np.newaxis]

    chirp_signal = np.asarray(chirp_signal)
    if chirp_signal.ndim > 1:
        chirp_signal = chirp_signal[:, 0]

    chirp_signal = chirp_signal * np.max(np.abs(recorded_signal))

    starts = []
    for i in range(recorded_signal.shape[1]):
        starts.append(cross_correlate(recorded_signal[:, i], chirp_signal, mode="full"))

    mean_start = int(np.mean(starts))
    chirp_end_index = mean_start + len(chirp_signal)

    return chirp_end_index, chirp_end_index / recorded_sample_rate


def find_interesting_audio_end(recording_file_path, start_index=0, plot=False,
                               trim_detection_plot_path=None, trim_audio_plot_path=None):

    signal, sample_rate = sf.read(recording_file_path)
    signal = np.asarray(signal)

    frame_ms = 25.0
    smoothing_ms = 200.0
    tail_seconds = 1.0
    threshold_multiplier = 3.0
    safety_margin_ms = 750.0

    if signal.ndim == 1:
        mono = signal.astype(np.float64)
    else:
        channel_rms = np.sqrt(np.mean(signal.astype(np.float64) ** 2, axis=0) + 1e-12)
        mono = signal[:, int(np.argmax(channel_rms))].astype(np.float64)

    n_samples = len(mono)
    if n_samples == 0:
        return 0, 0.0

    frame_length = max(1, int(round((frame_ms / 1000.0) * sample_rate)))
    smoothing_frames = max(1, int(round(smoothing_ms / frame_ms)))
    tail_frames = max(1, int(round((tail_seconds * sample_rate) / frame_length)))
    safety_margin_samples = int(round((safety_margin_ms / 1000.0) * sample_rate))

    n_frames = int(np.ceil(n_samples / frame_length))

    rms = np.empty(n_frames, dtype=np.float64)
    for i in range(n_frames):
        start = i * frame_length
        stop = min(n_samples, start + frame_length)
        frame = mono[start:stop]
        rms[i] = np.sqrt(np.mean(frame**2) + 1e-12)

    if smoothing_frames == 1:
        smoothed_rms = rms
    else:
        kernel = np.ones(smoothing_frames) / smoothing_frames
        smoothed_rms = np.convolve(rms, kernel, mode="same")

    tail = smoothed_rms[-tail_frames:]
    noise_floor = float(np.mean(tail))
    threshold = noise_floor * threshold_multiplier

    start_index = int(np.clip(start_index, 0, n_samples - 1))
    start_frame = max(0, start_index // frame_length)

    last_active_frame = None
    for frame_idx in range(n_frames - 1, start_frame - 1, -1):
        if smoothed_rms[frame_idx] > threshold:
            last_active_frame = frame_idx
            break

    if last_active_frame is None:
        end_sample = start_index
    else:
        frame_end = min(n_samples, (last_active_frame + 1) * frame_length)
        end_sample = min(n_samples, frame_end + safety_margin_samples)

    end_index = int(np.clip(end_sample - 1, 0, n_samples - 1))

    if plot:
        import matplotlib.pyplot as plt

        time_axis = (np.arange(n_frames) * frame_length) / sample_rate

        # ---- Plot 1 ----
        def plot_trim_detection():
            plot_lock.acquire()
            fig1, ax1 = plt.subplots()
            ax1.plot(time_axis, rms, alpha=0.35, color='tab:blue')
            ax1.plot(time_axis, smoothed_rms, color='tab:orange')
            ax1.axhline(threshold, linestyle="--", color='goldenrod')
            ax1.axvline(start_index / sample_rate, linestyle="--", color='tab:green')
            ax1.axvline(end_index / sample_rate, linestyle="--", color='tab:red')
            ax1.set_xlim(0.0, n_samples / sample_rate)
            fig1.legend(['RMS', 'Smoothed RMS', 'Threshold', 'Start Index', 'End Index'], loc='upper right')
            fig1.tight_layout()

            if trim_detection_plot_path:
                fig1.savefig(trim_detection_plot_path)
                
            plt.close(fig1)
            plot_lock.release()

        t1 = threading.Thread(target=plot_trim_detection)
        plot_threads.append(t1)
        t1.start()

        # ---- Plot 2 ----
        start_time_sec = start_index / sample_rate
        end_time_sec = end_index / sample_rate

        trimmed_mask = (time_axis >= start_time_sec) & (time_axis <= end_time_sec)
        trimmed_time_axis = time_axis[trimmed_mask] - start_time_sec

        def plot_trimmed_audio():
            plot_lock.acquire()
            fig2, ax2 = plt.subplots()
            ax2.plot(trimmed_time_axis, rms[trimmed_mask], alpha=0.35, color='tab:blue')
            ax2.plot(trimmed_time_axis, smoothed_rms[trimmed_mask], color='tab:orange')
            ax2.axhline(threshold, linestyle="--", color='goldenrod')
            ax2.set_xlim(0.0, end_time_sec - start_time_sec)
            fig2.legend(['RMS', 'Smoothed RMS', 'Threshold'], loc='upper right')
            fig2.tight_layout()

            if trim_audio_plot_path:
                fig2.savefig(trim_audio_plot_path)

            plt.close(fig2)
            plot_lock.release()
        
        t2 = threading.Thread(target=plot_trimmed_audio)
        plot_threads.append(t2)
        t2.start()

    return end_index, end_index / sample_rate

# --- Functions to play audio ---
def list_output_devices():
    """Prints all available audio output devices and their IDs."""
    print(sd.query_devices())


output_streams = [None, None]
def prepare_output_stream(data, fs, device_id, index):
     # Open the output stream
    output_streams[index] = sd.OutputStream(
        samplerate=fs,
        device=device_id,
        channels=data.shape[1] if data.ndim > 1 else 1,
        latency='low',
        dtype='float32',
    )

    # Start the output stream
    output_streams[index].start()

def play_audio_on_output_stream(data, index):
    # Check data is in float32 format
    if data.dtype != np.float32:
        data = data.astype(np.float32)

    # Write the audio data to the output stream
    output_streams[index].write(data)

def close_output_stream(index):
    if output_streams[index] is not None:
        output_streams[index].stop()
        output_streams[index].close()

# --- Functions to save and plot spikes ---
def save_files(arg_addresses, arg_timestamps, arg_compressed_spikes_path, arg_save_uncompressed, arg_uncompressed_spikes_path, arg_trim_audio_path, arg_data, arg_fs, arg_rel_path):
    try:
        # Create local copies of the arrays to avoid issues with threading and memory management
        addresses_local = np.copy(arg_addresses)
        timestamps_local = np.copy(arg_timestamps)

        # Save the trimmed audio and compressed spike data
        print(f"Saving files for {arg_rel_path}...")
        sf.write(arg_trim_audio_path, arg_data, arg_fs)
        AERzip.saveCompressedFile(addresses_local, timestamps_local, str(arg_compressed_spikes_path), overwrite=True)
        if arg_save_uncompressed:
            spike_file = SpikesFile(addresses_local, timestamps_local)
            Savers.save_AEDAT(spike_file, str(arg_uncompressed_spikes_path), settings)  # TODO: Removing ts_tick mult operation in save_AEDAT function?
    except Exception as e:
        print(Fore.RED + f"Error saving files for {arg_rel_path}: {e}")


def plot_sonogram(addresses, timestamps, sonogram_plot_path=None):
    addresses = np.copy(addresses) 
    timestamps = np.copy(timestamps) / 10**8  # Convert timestamps to seconds

    # ---- time normalization ----
    t_min = np.min(timestamps)
    t = timestamps - t_min

    # ---- fixed resolution (important for performance stability) ----
    n_bins = 500
    max_timestamp = np.max(timestamps)
    duration = max_timestamp - t_min
    bin_size = duration / n_bins if duration > 0 else 1.0
    inv_bin = 1.0 / bin_size

    # ---- binning (vectorized) ----
    bin_idx = np.minimum((t * inv_bin).astype(np.int32), n_bins - 1)
    bin_start_times = t_min + np.arange(n_bins) * bin_size

    # ---- number of channels ----
    n_addr = int(np.max(addresses)) + 1

    # ---- FASTEST PATH: bincount flattening ----
    flat_idx = addresses * n_bins + bin_idx
    sonogram = np.bincount(
        flat_idx,
        minlength=n_addr * n_bins
    ).reshape(n_addr, n_bins)

    # ---- plot (but DO NOT show) ----
    plot_lock.acquire()
    fig, ax = plt.subplots()

    im = ax.imshow(
        sonogram,
        aspect="auto",
        origin="lower",
        cmap="hot",
        extent=[
        bin_start_times[0],
        bin_start_times[-1] + bin_size,
        0,
        n_addr - 1]
    )

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Address")
    ax.set_title("Sonogram")

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    if sonogram_plot_path:
        fig.savefig(sonogram_plot_path)
    plt.close(fig)
    plot_lock.release()

## Selecting audio interfaces for audioplayback

In [4]:
okaer_audio_device = None
speakers_audio_device = None
if okaer_audio_device is None or speakers_audio_device is None:
    list_output_devices()
    okaer_audio_device = int(input("Please set the okaer_audio_device variable to the ID of your desired output device: "))
    speakers_audio_device = int(input("Please set the speakers_audio_device variable to the ID of your desired output device: "))

   0 Asignador de sonido Microsoft - Input, MME (2 in, 0 out)
>  1 Micrófono (Realtek(R) Audio), MME (2 in, 0 out)
   2 Asignador de sonido Microsoft - Output, MME (0 in, 2 out)
<  3 Realtek HD Audio 2nd output (Re, MME (0 in, 2 out)
   4 Altavoces (Realtek(R) Audio), MME (0 in, 2 out)
   5 Controlador primario de captura de sonido, Windows DirectSound (2 in, 0 out)
   6 Micrófono (Realtek(R) Audio), Windows DirectSound (2 in, 0 out)
   7 Controlador primario de sonido, Windows DirectSound (0 in, 2 out)
   8 Realtek HD Audio 2nd output (Realtek(R) Audio), Windows DirectSound (0 in, 2 out)
   9 Altavoces (Realtek(R) Audio), Windows DirectSound (0 in, 2 out)
  10 Realtek HD Audio 2nd output (Realtek(R) Audio), Windows WASAPI (0 in, 2 out)
  11 Altavoces (Realtek(R) Audio), Windows WASAPI (0 in, 2 out)
  12 Micrófono (Realtek(R) Audio), Windows WASAPI (2 in, 0 out)
  13 Headphones (Realtek HD Audio 2nd output), Windows WDM-KS (0 in, 2 out)
  14 Speakers (Realtek HD Audio output), Windows 

## Experiment: Select a folder for audio playback and spike monitoring

### Important: Set volume to 10%

In [ ]:

import threading
from colorama import Fore
import numpy as np
import time

# Monitor the inputs
INPUTS = ['port_a']  # Monitor only port_a where the CNAS outputs are sent. port_b is not used in this configuration
USB_TRANSFER_LENGTH = 64 * 1024

# Set USB transfer length and number of buffers
okaer.USB_TRANSFER_LENGTH = USB_TRANSFER_LENGTH

# Reset the okaerTool board before monitoring to ensure a clean state
okaer.reset_board(mode='internal')

# Set logger to ERROR level to reduce verbosity during processing
okaer.logger.setLevel(okt.logging.ERROR)

# Set up base directories
base_plot = '../Plots'
base_comp = '../Aedat files'
base_trim = '../Dataset (Trim)'

# Fast mode: keep the required board reconfiguration, but skip expensive extras by default
VERBOSE_LOGGING = True
SAVE_PLOTS = True
SAVE_UNCOMPRESSED = False
PLAY_ON_SPEAKERS = False

# Use fixed chirp reference file from the Auxiliary folder
chirp_file_path = str(Path('../Auxiliary/chirp_series.wav').resolve())

if not Path(chirp_file_path).exists():
    print(f'Chirp file not found: {chirp_file_path}. Aborting.')
else:
    print(f'Using chirp reference: {chirp_file_path}')
    program_start_time = time.time()

    # Select the folder containing the WAV files to be processed
    wav_folder_path = select_wav_folder()

    # Process each WAV file in the selected folder
    if wav_folder_path:
        wav_files = collect_wav_files(wav_folder_path)
        if not wav_files:
            print(f"No WAV files found in the selected folder: {wav_folder_path}")
        else:
            print(f"Found {len(wav_files)} WAV files in {wav_folder_path}")

            # Create subfolder names based on the selected folder
            wav_root = Path(wav_folder_path).resolve()
            folder_name = wav_root.name
            target_plot = Path(base_plot) / folder_name
            target_comp = Path(base_comp) / folder_name
            target_audio_trim = Path(base_trim) / folder_name
            target_plot.mkdir(parents=True, exist_ok=True)
            target_comp.mkdir(parents=True, exist_ok=True)
            target_audio_trim.mkdir(parents=True, exist_ok=True)

            save_files_threads = []
            plot_threads = []
            plot_lock = threading.Lock()
            last_fs = None

            i = 1
            n_files = len(wav_files)
            for wav_file in wav_files:
                try:
                    wav_path = Path(wav_file)
                    rel_path = wav_path.relative_to(wav_root)
                    file_base = wav_path.stem
                    output_plot_dir = target_plot / rel_path.parent
                    output_comp_dir = target_comp / rel_path.parent
                    output_audio_trim_dir = target_audio_trim / rel_path.parent
                    output_plot_dir.mkdir(parents=True, exist_ok=True)
                    output_comp_dir.mkdir(parents=True, exist_ok=True)
                    output_audio_trim_dir.mkdir(parents=True, exist_ok=True)

                    # Check if output files already exist and skip processing if they do
                    sonogram_path = output_plot_dir / f'{file_base}_sonogram.png'
                    compressed_spikes_path = output_comp_dir / f'{file_base}_spikes.aedat'
                    trim_detection_plot_path = output_plot_dir / f'{file_base}_trim_detection.png'
                    trim_audio_plot_path = output_plot_dir / f'{file_base}_trimmed.png'
                    trim_audio_path = output_audio_trim_dir / f'{file_base}_trimmed.wav'
                    uncompressed_spikes_path = output_comp_dir / f'{file_base}_spikes_uncompressed'
                    if SAVE_UNCOMPRESSED:
                        if sonogram_path.exists() and compressed_spikes_path.exists() and uncompressed_spikes_path.exists() and trim_detection_plot_path.exists() and trim_audio_plot_path.exists() and trim_audio_path.exists():
                            print(f"Skipping {rel_path}: Output files already exist")
                            continue
                    else:
                        if sonogram_path.exists() and compressed_spikes_path.exists() and trim_detection_plot_path.exists() and trim_audio_plot_path.exists() and trim_audio_path.exists():
                            print(f"Skipping {rel_path}: Output files already exist")
                            continue

                    print(f"\n--- Processing audio file: {rel_path} ({i} of {n_files}) ---")
                    i += 1

                    # Load the WAV file
                    data, fs = sf.read(str(wav_path))

                    # Prepare the output streams for both Okaer and speakers (if enabled) to reduce startup latency
                    if last_fs is None or last_fs != fs:
                        for index in range(len(output_streams)):
                            close_output_stream(index)
                                
                        prepare_output_stream(data, fs, okaer_audio_device, index=0)
                        if PLAY_ON_SPEAKERS:
                            prepare_output_stream(data, fs, speakers_audio_device, index=1)
                        last_fs = fs

                    # Detect chirp end (trim start)
                    start_time = time.time()
                    trim_start_index, trim_start_time = find_chirp_end(str(wav_path), chirp_file_path)
                    trim_start_index = int(trim_start_index)  # Removing the fractional part to ensure it's an integer index
                    end_time = time.time()
                    print(f"Chirp detection completed in {end_time - start_time:.2f} seconds.")

                    # Detect the end of the target segment (trim end)
                    start_time = time.time()
                    trim_end_index, trim_end_time = find_interesting_audio_end(str(wav_path), start_index=trim_start_index, plot=SAVE_PLOTS, trim_detection_plot_path=str(trim_detection_plot_path), trim_audio_plot_path=str(trim_audio_plot_path))
                    end_time = time.time()
                    print(f"Interesting audio detection completed in {end_time - start_time:.2f} seconds.")

                    # Trim the audio data to the target segment
                    data = data[trim_start_index:trim_end_index]
                    trimmed_duration_sec = len(data) / fs

                    # Send it to the cochlea for spike generation and monitor the spikes in a separate thread    
                    #reset_and_configure_okaer()  # Ensure the board is reset and configured before monitoring
                    okaer.reset_board(mode='internal')                
                    synchronization_event = threading.Event()
                    
                    spikes_result = {}
                    def monitor_spikes():
                        spikes_result['spikes'] = okaer.monitor(inputs=INPUTS, duration=trimmed_duration_sec)
                        
                    def audio_playback(data, index):
                        synchronization_event.wait()
                        play_audio_on_output_stream(data, index=index)

                    print("Starting audio playback and spike monitoring...")
                    monitor_thread = threading.Thread(target=monitor_spikes)
                    okaer_play_audio_thread = threading.Thread(target=audio_playback, args=(data, 0))
                    if PLAY_ON_SPEAKERS:
                        speakers_play_audio_thread = threading.Thread(target=audio_playback, args=(data, 1))

                    monitor_thread.start()
                    okaer_play_audio_thread.start()
                    if PLAY_ON_SPEAKERS:
                        speakers_play_audio_thread.start()

                    while not okaer.is_monitoring or okaer.usb_read_thread is None or not okaer.usb_read_thread.is_alive(): # Wait for okaer monitoring to be ready before starting playback
                        time.sleep(0.001)
                    synchronization_event.set()  # Signal both playback threads to start simultaneously

                    okaer_play_audio_thread.join()  # Wait for the audio playback to finish before proceeding
                    monitor_thread.join()  # Wait for the monitoring to finish before proceeding
                    
                    # Getting the spikes result after both threads have completed
                    spikes = spikes_result.get('spikes')
                    if not spikes:
                        print("No spikes were recorded for %s. Skipping.", rel_path)
                        continue

                    print("Spike monitoring completed. Processing results...")

                    addresses = np.array(spikes[0].addresses)
                    timestamps = np.array(spikes[0].timestamps)
                    addr_len = len(addresses)
                    ts_len = len(timestamps)
                    if addr_len != ts_len:
                        print("Mismatch: %d addresses vs %d timestamps!", addr_len, ts_len)
                        continue

                    # Shifting the timestamps to align with the trimmed audio segment
                    print(f"Trimmed duration = {trimmed_duration_sec:.2f} seconds")
                    trimmed_duration_eq = trimmed_duration_sec * 10**8
                    seg_max_timestamp = np.max(timestamps)
                    seg_min_timestamp = seg_max_timestamp - trimmed_duration_eq

                    if seg_min_timestamp < 0:
                        print(Fore.RED + f"Warning: The segment is shorter than the trimmed audio duration ({trimmed_duration_sec:.6f}s)")

                    valid_indices = np.where(timestamps >= seg_min_timestamp)[0]
                    timestamps = timestamps[valid_indices]
                    addresses = addresses[valid_indices]

                    # Launching threads to save the spike files, trimmed audio and plots (if enabled)
                    save_files_thread = threading.Thread(target=save_files, args=(addresses, timestamps, compressed_spikes_path, SAVE_UNCOMPRESSED, uncompressed_spikes_path, trim_audio_path, data, fs, rel_path))
                    save_files_threads.append(save_files_thread)
                    save_files_thread.start()  # Do not wait for it to finish, let it run in the background while processing the next file
    
                    if SAVE_PLOTS:
                        plot_sonogram_thread = threading.Thread(target=plot_sonogram, args=(addresses, timestamps, str(sonogram_path)))
                        plot_threads.append(plot_sonogram_thread)
                        plot_sonogram_thread.start()

                except Exception as e:
                    print(Fore.RED +f"Error processing {rel_path}: {e}")
                    continue

            # Close the output streams and wait for threads to finish before exiting the program
            for index in range(len(output_streams)):
                close_output_stream(index)

            for thread in save_files_threads:
                thread.join()
            for thread in plot_threads:
                thread.join()

            elapsed_time = time.time() - program_start_time
            print(f"Finished processing {len(wav_files)} WAV files.")
            print(f"Total processing time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
    else:
        print("No folder was selected.")


Using chirp reference: C:\Users\alvco\Desktop\Manchester2026\Auxiliary\chirp_series.wav
Found 460 WAV files in C:/Users/alvco/Desktop/Manchester2026/Dataset/1

--- Processing audio file: r_0.wav (1 of 460) ---
Chirp detection completed in 0.04 seconds.
Interesting audio detection completed in 0.02 seconds.
Starting audio playback and spike monitoring...
Spike monitoring completed. Processing results...
Trimmed duration = 3.34 seconds

--- Processing audio file: r_1.wav (2 of 460) ---
Saving files for r_0.wav...
[  4685658   4685867   4686290 ... 338780777 338782647 338784445]
Chirp detection completed in 0.18 seconds.
Interesting audio detection completed in 0.83 seconds.
Starting audio playback and spike monitoring...
Spike monitoring completed. Processing results...
Trimmed duration = 3.11 seconds

--- Processing audio file: r_10.wav (3 of 460) ---
Saving files for r_1.wav...
[  3766549   3766665   3767048 ... 314688143 314688155 314688941]
Chirp detection completed in 0.21 seconds.
